# AgriPrice-IQ: Feature Engineering
This notebook transforms our cleaned historical retail price data into a robust feature matrix optimized for LightGBM time-series forecasting.

### Objectives:
1. Extract temporal features (Month, Year, Quarter, and cyclical Sine/Cosine transformations).
2. Generate lag features (historical prices from 1, 3, 6, and 12 months prior).
3. Compute rolling window statistics (rolling mean and standard deviation over 3 and 6-month windows).
4. Clean missing values and export the modeling-ready dataset to SQLite.

In [2]:
import sqlite3
import pandas as pd
import numpy as np

# Connect to database and load clean retail data
conn = sqlite3.connect('../database/agriprice_iq.db')
df = pd.read_sql("SELECT * FROM retail_prices;", conn)
conn.close()

# Ensure chronological sorting per group (essential for correct lag generation)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['province', 'product_name', 'date']).reset_index(drop=True)

print(f"Loaded dataset shape: {df.shape}")
display(df.head())

Loaded dataset shape: (74369, 6)


,product_name,province,date_raw,price,date,category
0,"Almonds, 200 grams 6",Alberta,January 2021,5.21,2021-01-01,Pantry Staples
1,"Almonds, 200 grams 6",Alberta,February 2021,5.40,2021-02-01,Pantry Staples
2,"Almonds, 200 grams 6",Alberta,March 2021,5.38,2021-03-01,Pantry Staples
3,"Almonds, 200 grams 6",Alberta,April 2021,5.37,2021-04-01,Pantry Staples
4,"Almonds, 200 grams 6",Alberta,May 2021,5.47,2021-05-01,Pantry Staples


In [4]:
# Extract standard date features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter

# Cyclical encoding for months to capture seasonal continuity (e.g., transition from Dec to Jan)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

print("Temporal and cyclical features added successfully!")

Temporal and cyclical features added successfully!


In [8]:
# Group by geographical region and product to prevent cross-contamination of lags across different items/regions
group_cols = ['province', 'product_name']

# Create lag features for 1, 3, 6, and 12 months prior
lags = [1, 3, 6, 12]
for lag in lags:
    df[f'lag_{lag}'] = df.groupby(group_cols)['price'].shift(lag)

print("Lag features generated successfully!")

Lag features generated successfully!


In [9]:
# Compute rolling window features (3-month and 6-month rolling averages/std)
# shift(1) ensures we use past data only (preventing data leakage)
df['rolling_mean_3m'] = df.groupby(group_cols)['price'].transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
df['rolling_std_3m'] = df.groupby(group_cols)['price'].transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).std()).fillna(0)

df['rolling_mean_6m'] = df.groupby(group_cols)['price'].transform(lambda x: x.shift(1).rolling(window=6, min_periods=1).mean())
df['rolling_std_6m'] = df.groupby(group_cols)['price'].transform(lambda x: x.shift(1).rolling(window=6, min_periods=1).std()).fillna(0)

print("Rolling statistics generated successfully!")

Rolling statistics generated successfully!


In [10]:
# Drop rows with NaN values created by initial lag shifts (e.g., the first 12 months)
df_model_ready = df.dropna().reset_index(drop=True)

print(f"Shape before cleaning NaNs: {df.shape}")
print(f"Shape after cleaning NaNs (Model-Ready): {df_model_ready.shape}")

# Save the feature-engineered dataset back into SQLite as a new table
conn = sqlite3.connect('../database/agriprice_iq.db')
df_model_ready.to_sql('model_features', conn, if_exists='replace', index=False)
conn.close()

print("Model-ready feature set successfully saved to SQLite table 'model_features'!")

Shape before cleaning NaNs: (74369, 19)
Shape after cleaning NaNs (Model-Ready): (61181, 19)
Model-ready feature set successfully saved to SQLite table 'model_features'!


## Summary
We have successfully constructed a rich feature matrix containing temporal markers, autoregressive lags, and momentum-based rolling statistics. The dataset is now clean, free of data leakage, and stored securely in the `model_features` SQLite table, ready for LightGBM quantile regression.